# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!pip install -U langchain langchain-community langchain-openai

  Using cached langchain-1.0.8-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_openai-1.0.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchain_core-1.0.7-py3-none-any.whl.metadata (3.6 kB)
  Using cached langchain_text_splitters-1.0.0-py3-none-any.whl.metadata (2.6 kB)
Using cached langchain-1.0.8-py3-none-any.whl (93 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached langchain_openai-1.0.3-py3-none-any.whl (82 kB)
Using cached langchain_core-1.0.7-py3-none-any.whl (472 kB)
Using cached langchain_text_splitters-1.0.0-py3-none-any.whl (33 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.80
    Uninstalling langchain-core-0.3.80:
      Successfully uninstalled langchain-core-0.3.80
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.11
    Uninstalling langch

In [2]:
!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken langchain-community

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-core 0.3.80 which is incompatible.
langchain-classic 1.0.0 requires langchain-core<2.0.0,>=1.0.0, but you have langchain-core 0.3.80 which is incompatible.
langchain-classic 1.0.0 requires langchain-text-splitters<2.0.0,>=1.0.0, but you have langchain-text-splitters 0.3.11 which is incompatible.


In [3]:
# !pip install -U langchain langchain-community langchain-openai

In [4]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

In [5]:
from langchain_community.vectorstores.elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial.

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [6]:
# Prompt securely for credentials
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")
ELASTIC_API_KEY = getpass("Elastic API Key: ")
OPENAI_API_KEY = getpass("OpenAI API Key: ")

print("Secrets loaded successfully ✔")


Elastic Cloud ID: ··········
Elastic API Key: ··········
OpenAI API Key: ··········
Secrets loaded successfully ✔


In [7]:
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="workplace_documents", # give it a meaningful name, like 'workplace_documents'
    embedding=embeddings,
)

/tmp/ipython-input-5982053.py:3: LangChainPendingDeprecationWarning: The class `ElasticsearchStore` will be deprecated in a future version. Use :class:`~Use class in langchain-elasticsearch package` instead.
  vectorstore = ElasticsearchStore(


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [8]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [9]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dateutil.parser import parse

def metadata_func(record: dict, metadata: dict) -> dict:
    metadata['name'] = record.get('name', 'unknown-source')
    metadata['summary'] = record.get('summary', '')
    metadata['url'] = record.get('url', '')
    metadata['category'] = record.get('category', '')

    updated_at = record.get('updated_at')
    if updated_at and updated_at.strip():
        try:
            dt = parse(updated_at)
            metadata['updated_at'] = dt.isoformat()
        except Exception as e:
            print(f"Warning: Failed to parse date '{updated_at}' - {e}")
            metadata.pop('updated_at', None)
    else:
        metadata.pop('updated_at', None)

    print(f"Processing record, updated_at: {updated_at} -> metadata: {metadata.get('updated_at', 'Not set')}")
    return metadata




In [15]:
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000, chunk_overlap=100 #define chunk size and chunk overlap
)




docs = loader.load_and_split(text_splitter=text_splitter)

for i, doc in enumerate(docs[:3]):
    print(f"Doc {i+1} metadata:", doc.metadata)

for doc in docs[:3]:
    print(doc.metadata)  # Should include 'name'


for doc in docs:
    if 'name' not in doc.metadata or not doc.metadata['name']:
        doc.metadata['name'] = "unknown-source"
vectorstore = ElasticsearchStore.from_documents(
    docs,                # these docs with metadata!
    embeddings,
    index_name="adriari4",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)




Processing record, updated_at: 2020-03-01 -> metadata: 2020-03-01T00:00:00
Processing record, updated_at: 2022-04-29 -> metadata: 2022-04-29T00:00:00
Processing record, updated_at: 2023-05-01 -> metadata: 2023-05-01T00:00:00
Processing record, updated_at: 2023-04-15 -> metadata: 2023-04-15T00:00:00
Processing record, updated_at: 2018-04-16 -> metadata: 2018-04-16T00:00:00
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Processing record, updated_at: None -> metadata: Not set
Doc 1 metadata: {'source': '/content/temp.json', 'seq_n

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [17]:
vectorstore = ElasticsearchStore.from_documents(
    docs,                # these docs with metadata!
    embeddings,
    index_name="adriari4",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [18]:
from langchain.llms import OpenAI
from langchain.vectorstores.elasticsearch import ElasticsearchStore
from langchain.retrievers import MultiQueryRetriever
from langchain.schema.runnable import Runnable, RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document
import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

# Create vectorstore from documents (docs, embeddings, ELASTIC_* keys assumed defined)
vectorstore = ElasticsearchStore.from_documents(
    docs,
    embeddings,
    index_name="adriari4",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

# Initialize OpenAI LLM
llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

# Create MultiQueryRetriever from vectorstore retriever + LLM
retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Prompt templates
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible.

    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)

# Runnable class for combining docs into a single string
class CombineDocumentsRunnable(Runnable):
    def invoke(self, docs, *args, **kwargs):
        for doc in docs:
            if 'name' not in doc.metadata or not doc.metadata['name']:
                doc.metadata['name'] = "unknown-source"
        doc_strings = [format_document(doc, LLM_DOCUMENT_PROMPT) for doc in docs]
        return "\n\n".join(doc_strings)




_combine_documents = CombineDocumentsRunnable()

# Build the runnable chain
_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

# Ask your question
ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)


INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team at NASA operate?', '3. What are the responsibilities of the sales team at NASA?']


---- Answer ----
The NASA sales team is a part of the Americas region in the sales organization. It is responsible for serving customers in North America and South America, with two Area Vice-Presidents, Laura Martinez and Gary Johnson. The team works closely with other departments to identify and pursue new business opportunities, nurture existing client relationships, and ensure customer satisfaction.


# Task

**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [19]:
answer = chain.invoke("What is the NASA sales team and how do they operate?")

print("---- Chain-of-Thought Answer ----")
print(answer)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. How does the NASA sales team function and what are their operations?', '2. Can you explain the operations of the NASA sales team and their role?', '3. What are the functions and operations of the NASA sales team?']


---- Chain-of-Thought Answer ----
The NASA sales team is a part of the Americas region in the sales organization. It is responsible for serving customers and achieving business objectives in North and South America. The team is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. They work closely with dedicated account managers, sales representatives, and support staff to identify and pursue new business opportunities, nurture existing client relationships, and ensure customer satisfaction. The team also collaborates with other departments, such as marketing, product development, and customer support, to consistently deliver high-quality products and services to clients.


In [20]:
answer = chain.invoke("How does NASA approach sustainability and environmental impact?")

print("---- Chain-of-Thought Answer ----")
print(answer)

INFO:langchain.retrievers.multi_query:Generated queries: ["1. What is NASA's approach to sustainability and environmental impact?", '2. How does NASA address sustainability and its impact on the environment?', '3. In what ways does NASA tackle sustainability and its effects on the environment?']


---- Chain-of-Thought Answer ----

As an assistant for question-answering tasks, I am not able to provide information on specific company practices or policies. However, based on the retrieved context, it appears that NASA, which stands for North America South America region, is a sales region within the Americas division of the company. It is led by two Area Vice-Presidents, Laura Martinez and Gary Johnson, who are responsible for managing the sales team in North America and South America respectively. The sales team in this region is focused on identifying and pursuing new business opportunities, nurturing existing client relationships, and ensuring customer satisfaction. While the retrieved context does not mention anything about sustainability or environmental impact, it is possible that NASA, along with other regional sales teams, may have initiatives or strategies in place to address these issues. However, without further information, I am unable to provide a more specific answer

In [21]:
answer = chain.invoke("How many times did the astronauts reach the Moon?")

print("---- Chain-of-Thought Answer ----")
print(answer)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the number of times that the astronauts successfully landed on the Moon?', '2. Can you tell me the frequency of astronauts reaching the Moon?', '3. How often did the astronauts successfully complete a mission to the Moon?']


---- Chain-of-Thought Answer ----
I'm sorry, I don't know the answer to that question as it is not mentioned in the provided context. The context only discusses the sales organization and a work from home policy update.


In [23]:
answer = chain.invoke("What is SpaceX?")

print("---- Chain-of-Thought Answer ----")
print(answer)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on SpaceX?', '2. How would you describe SpaceX?', '3. What are the key features of SpaceX?']


---- Chain-of-Thought Answer ----
I'm sorry, I don't know the answer to that question. My purpose is to assist with question-answering tasks, and I don't have access to information outside of the provided context. Is there something else I can help you with?


In [24]:
answer = chain.invoke("What has the most important missions been of NASA?")

print("---- Chain-of-Thought Answer ----")
print(answer)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the key objectives that NASA has focused on in its missions?', '2. Can you provide a list of the most significant missions undertaken by NASA?', "3. How has NASA's mission priorities evolved over the years and what are the most notable ones?"]


---- Chain-of-Thought Answer ----

NASA's most important missions have been to increase revenue, expand market share, and strengthen customer relationships in their target markets. They have also focused on launching new products and services in high-demand market segments, optimizing their product/service portfolio, and enhancing post-sales support and customer service to improve customer satisfaction. Additionally, NASA has prioritized expanding into new markets and industries, strengthening relationships with key accounts and strategic partners, and developing tailored offerings for different customer segments based on their needs and preferences.


In [25]:
answer = chain.invoke("Who is Neil Amstrong?")

print("---- Chain-of-Thought Answer ----")
print(answer)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the background of Neil Armstrong?', "2. Can you tell me about Neil Armstrong's achievements?", '3. Who is the famous astronaut, Neil Armstrong?']


---- Chain-of-Thought Answer ----
I'm sorry, I don't know who Neil Armstrong is. However, based on the context provided, I can tell you that he is not mentioned in the Sales Organization Overview. Neil Armstrong was an American astronaut and the first person to walk on the moon. He is not related to the sales organization mentioned in the context.
